<a href="https://colab.research.google.com/github/nsamuelreddy/loan_approval_prediction/blob/main/xgboost(loan).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from scipy.stats import randint,uniform

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
# Load the dataset directly with the correct filename
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "architsharma01/loan-approval-prediction-dataset",
    "loan_approval_dataset.csv"  # Correct filename
)

DATA PREPROCESSING

In [ ]:
 # to remove white space before column name
df.columns = df.columns.str.strip()

In [ ]:
print(df.head())
print( )
a=df.columns
print("Columns are \n",a[:])
print( )
print("Sum of null values \n",df.isnull().sum())

   loan_id  no_of_dependents      education self_employed  income_annum  \
0        1                 2       Graduate            No       9600000   
1        2                 0   Not Graduate           Yes       4100000   
2        3                 3       Graduate            No       9100000   
3        4                 3       Graduate            No       8200000   
4        5                 5   Not Graduate           Yes       9800000   

   loan_amount  loan_term  cibil_score  residential_assets_value  \
0     29900000         12          778                   2400000   
1     12200000          8          417                   2700000   
2     29700000         20          506                   7100000   
3     30700000          8          467                  18200000   
4     24200000         20          382                  12400000   

   commercial_assets_value  luxury_assets_value  bank_asset_value loan_status  
0                 17600000             22700000           80

In [ ]:
from sklearn.preprocessing import LabelEncoder

for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,0,0,9600000,29900000,12,778,2400000,17600000,22700000,8000000,0
1,2,0,1,1,4100000,12200000,8,417,2700000,2200000,8800000,3300000,1
2,3,3,0,0,9100000,29700000,20,506,7100000,4500000,33300000,12800000,1
3,4,3,0,0,8200000,30700000,8,467,18200000,3300000,23300000,7900000,1
4,5,5,1,1,9800000,24200000,20,382,12400000,8200000,29400000,5000000,1


In [ ]:
df.describe()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
count,4269.000000,4269.000000,4269.000000,4269.000000,4.269000e+03,4.269000e+03,4269.000000,4269.000000,4.269000e+03,4.269000e+03,4.269000e+03,4.269000e+03,4269.000000
mean,2135.000000,2.498712,0.497775,0.503631,5.059124e+06,1.513345e+07,10.900445,599.936051,7.472617e+06,4.973155e+06,1.512631e+07,4.976692e+06,0.377840
std,1232.498479,1.695910,0.500054,0.500045,2.806840e+06,9.043363e+06,5.709187,172.430401,6.503637e+06,4.388966e+06,9.103754e+06,3.250185e+06,0.484904
min,1.000000,0.000000,0.000000,0.000000,2.000000e+05,3.000000e+05,2.000000,300.000000,-1.000000e+05,0.000000e+00,3.000000e+05,0.000000e+00,0.000000
25%,1068.000000,1.000000,0.000000,0.000000,2.700000e+06,7.700000e+06,6.000000,453.000000,2.200000e+06,1.300000e+06,7.500000e+06,2.300000e+06,0.000000
50%,2135.000000,3.000000,0.000000,1.000000,5.100000e+06,1.450000e+07,10.000000,600.000000,5.600000e+06,3.700000e+06,1.460000e+07,4.600000e+06,0.000000
75%,3202.000000,4.000000,1.000000,1.000000,7.500000e+06,2.150000e+07,16.000000,748.000000,1.130000e+07,7.600000e+06,2.170000e+07,7.100000e+06,1.000000
max,4269.000000,5.000000,1.000000,1.000000,9.900000e+06,3.950000e+07,20.000000,900.000000,2.910000e+07,1.940000e+07,3.920000e+07,1.470000e+07,1.000000


In [ ]:
df.corr()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
loan_id,1.000000,0.005326,-0.015536,0.001745,0.012592,0.008170,0.009809,0.016323,0.020936,0.018595,-0.000862,0.010765,-0.017685
no_of_dependents,0.005326,1.000000,-0.002697,0.000765,0.007266,-0.003366,-0.020111,-0.009998,0.007376,-0.001531,0.002817,0.011163,0.018114
education,-0.015536,-0.002697,1.000000,0.023224,-0.011625,-0.010631,0.008417,0.004649,-0.010930,0.006763,-0.012471,-0.009424,0.004918
self_employed,0.001745,0.000765,0.023224,1.000000,0.002368,0.001450,0.004107,-0.004866,0.006144,-0.017998,0.004413,-0.000215,-0.000345
income_annum,0.012592,0.007266,-0.011625,0.002368,1.000000,0.927470,0.011488,-0.023034,0.636841,0.640328,0.929145,0.851093,0.015189
loan_amount,0.008170,-0.003366,-0.010631,0.001450,0.927470,1.000000,0.008437,-0.017035,0.594596,0.603188,0.860914,0.788122,-0.016150
loan_term,0.009809,-0.020111,0.008417,0.004107,0.011488,0.008437,1.000000,0.007810,0.008016,-0.005478,0.012490,0.017177,0.113036
cibil_score,0.016323,-0.009998,0.004649,-0.004866,-0.023034,-0.017035,0.007810,1.000000,-0.019947,-0.003769,-0.028618,-0.015478,-0.770518
residential_assets_value,0.020936,0.007376,-0.010930,0.006144,0.636841,0.594596,0.008016,-0.019947,1.000000,0.414786,0.590932,0.527418,0.014367
commercial_assets_value,0.018595,-0.001531,0.006763,-0.017998,0.640328,0.603188,-0.005478,-0.003769,0.414786,1.000000,0.591128,0.548576,-0.008246


In [ ]:
df.drop('loan_id',axis=1,inplace=True)

Data splitting to train,test

In [ ]:
x=df.drop('loan_status',axis=1)
y=df['loan_status']

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

Create the model

In [ ]:
model=XGBClassifier(
    randome_state=42,
    eval_metric="logloss"
)

HYPER PARAMETERS

In [ ]:
params={
    "n_estimators":randint(50,300),
    "max_depth":randint(3,10),
    "learning_rate":uniform(0.01,0.29),
    "subsample":uniform(0.6,0.4),
    "colsample_bytree":uniform(0.6,0.4)
}

RandomizedSearch

In [ ]:
random=RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    random_state=42,
    cv=5,
    n_iter=20,
    scoring="accuracy",
    n_jobs=-1
)

In [ ]:
random.fit(x_train,y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:25:33] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "randome_state" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


RandomizedSearchCV(cv=5,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=True,
                                           eval_metric='logloss',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_const...
                                        'learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x79e4467c44d0>,
                                        'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x79e4467c4620>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x79e4467c45f0>,
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x79e4467c4740>},
                   random_state=42, scoring='accuracy')

In [ ]:
best_model=random.best_estimator_

In [ ]:
predict=best_model.predict(x_test)

Accuracy

In [ ]:
accuracy_best=accuracy_score(y_test,predict)
print(accuracy_best)

0.9765807962529274


Confusion Matrix

In [ ]:
cm=confusion_matrix(y_test,predict)
cm

array([[525,  11],
       [  9, 309]])

Best Parameters

In [ ]:
best_parameters=random.best_params_
best_score=random.best_score_
print("Best Parameters are :",best_parameters)
print("Best Score is ",best_score)

Best Parameters are : {'colsample_bytree': np.float64(0.8439986631130484), 'learning_rate': np.float64(0.25162652440348765), 'max_depth': 5, 'n_estimators': 255, 'subsample': np.float64(0.7564242430292963)}
Best Score is  0.9815519765739384


Feature Importance

In [ ]:
importance=pd.DataFrame({
    'feature':x.columns,
    'Importance':best_model.feature_importances_
})
print("Data about Feauture Importance is \n",importance)

print("Total Accuracy ",accuracy_best)

Data about Feauture Importance is 
                      feature  Importance
0           no_of_dependents    0.014533
1                  education    0.008231
2              self_employed    0.012958
3               income_annum    0.031440
4                loan_amount    0.033441
5                  loan_term    0.201710
6                cibil_score    0.637404
7   residential_assets_value    0.017787
8    commercial_assets_value    0.013709
9        luxury_assets_value    0.016914
10          bank_asset_value    0.011873
Total Accuracy  0.9765807962529274


Save Model

In [ ]:
import joblib
joblib.dump(best_model,"loan_approval_xgboost.pkl")

['loan_approval_xgboost.pkl']